In [1]:
# Import necessary libraries
import pandas as pd
from spotapi import PublicPlaylist
import json

In [2]:
# Load the data
df = pd.read_csv('tracks.csv')

# Display the column names to show the features available in the dataset
print("Column names:")
print(df.columns.tolist())

# Make sure the 'release_date' consists of only the year part
df["release_date"] = df["release_date"].str[:4].astype(int)

# Take only songs from 2000 onwards
df = df[(df['release_date'] >= 2000)]

# Remove repeating tracks
df = df.drop_duplicates(subset=['artists', 'name'])

# Number of samples
num_samples = df.shape[0]

# Display the number of samples after cleaning
print(f"Number of samples: {num_samples}")


Column names:
['id', 'name', 'popularity', 'duration_ms', 'explicit', 'artists', 'id_artists', 'release_date', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']
Number of samples: 193467


In [3]:
def extract_song_name_artist(playlists):
    """
    Extracts unique song names and artists 
    from a list of Spotify playlist IDs.
    """

    unique_tracks = {}

    for playlist_id in playlists:

        playlist = PublicPlaylist(playlist_id)
        playlist_info = playlist.get_playlist_info()

        try:
            items = playlist_info["data"]["playlistV2"]["content"]["items"]
        except KeyError:
            print(f"Skipping playlist (no content): {playlist_id}")
            continue
        
        for item in items:

            track_data = item["itemV2"]["data"]

            # Extract track ID
            track_uri = track_data["uri"]
            track_id = track_uri.split(":")[-1]

            # Skip duplicates
            if track_id in unique_tracks:
                continue

            # Extract song info
            song_name = track_data["name"]

            artist_name = (
                track_data["artists"]["items"][0]["profile"]["name"]
            )

            # Store unique track
            unique_tracks[track_id] = {
                "id": track_id,
                "name": song_name,
                "artist": artist_name
            }
        
    return list(unique_tracks.values())

In [4]:
def match_playlist_tracks(df, songs_list):
    """
    Match pre-extracted songs (id, name, artist)
    with dataset using ID first, then fallback.
    """

    matched_rows = []

    # Preprocess dataset once
    df_copy = df.copy()
    df_copy["name_lower"] = df_copy["name"].str.lower()
    df_copy["artists_lower"] = df_copy["artists"].str.lower()

    for track in songs_list:

        track_id = track["id"]

        # -------- ID MATCH --------
        id_match = df_copy[df_copy["id"] == track_id]

        if not id_match.empty:
            matched_rows.append(id_match)
            continue

        # -------- FALLBACK --------
        song_name = track["name"].lower()
        artist_name = track["artist"].lower()

        fallback_match = df_copy[
            (df_copy["name_lower"] == song_name) &
            (df_copy["artists_lower"].str.contains(artist_name, na=False))
        ]

        if not fallback_match.empty:
            matched_rows.append(fallback_match)

    if not matched_rows:
        return pd.DataFrame()

    result = pd.concat(matched_rows, ignore_index=True)
    result = result.drop_duplicates(subset="id")

    return result.drop(columns=["name_lower", "artists_lower"])

In [5]:
happy_playlists = [
    "37i9dQZF1EIeEZPgsd7pko",
    "37i9dQZF1EIgG2NEOhqsD7",
    "37i9dQZF1EIgNoWOvbnUCk",
    "37i9dQZF1EIhA3apoIYlV8",
    "37i9dQZF1EVJSvZp5AOML2",
    "37i9dQZF1EIdSqVcLpmJGJ",
    "37i9dQZF1EIfAiavURxjpo",
    "7GhawGpb43Ctkq3PRP1fOL",
    "37i9dQZF1EIhPBBOvFSqO1",
    "37i9dQZF1EIg50fSlCioBM",
    "37i9dQZF1EIcHq5v2wdaUn",
    "37i9dQZF1EIdMgPUiweOJE"
]

happy_songs = extract_song_name_artist(happy_playlists)
print(f"Collected {len(happy_songs)} unique happy songs")

happy_songs_df = match_playlist_tracks(df, happy_songs)
print(f"Happy songs - Number of samples: {len(happy_songs_df)}")

Collected 226 unique happy songs
Happy songs - Number of samples: 104


In [6]:
sad_playlists = [
    "37i9dQZF1EIg85EO6f7KwU",
    "37i9dQZF1EIhca1Hr096mb",
    "37i9dQZF1EIeODNDegVpao",
    "37i9dQZF1EIfv2exTKzl3M",
    "37i9dQZF1EIg6gLNLe52Bd",
    "37i9dQZF1EIcxHInSBQ4YQ"
    "5DVUEqRL1EV8I9n65eBaAw",
    "37i9dQZF1EIgQSl88QXPwP",
    "37i9dQZF1EIhkYYl6OyR5D",
    "2ZnAWYy4AOs8tpRUCGF6Py",
    "2IaFYYY5wZ1XjrGYQ759EH",
    "3jCLr8DeI6HWHjTpslJXf8",
    "12mnv0AK3p2ngWfhTqraRW",
    "5mLIo3cu4cJRcdjzfMpQZt"
]

sad_songs = extract_song_name_artist(sad_playlists)
print(f"Collected {len(sad_songs)} unique sad songs")

sad_songs_df = match_playlist_tracks(df, sad_songs)
print(f"Sad songs - Number of samples: {len(sad_songs_df)}")

Skipping playlist (no content): 37i9dQZF1EIcxHInSBQ4YQ5DVUEqRL1EV8I9n65eBaAw
Collected 240 unique sad songs
Sad songs - Number of samples: 99


In [7]:
# Check if there are any songs that are in both happy and sad playlists
overlap = pd.merge(happy_songs_df, sad_songs_df, on=["name", "artists"], how="inner")
print(f"Number of songs in both happy and sad playlists: {len(overlap)}")

# Remove overlapping songs from happy dataset
overlap_ids = overlap["id_x"].unique()
happy_songs_df = happy_songs_df[~happy_songs_df["id"].isin(overlap_ids)]

Number of songs in both happy and sad playlists: 1


In [8]:
# Download the datasets as CSV files with labels

# Set labels for happy and sad songs
happy_songs_df["label"] = "happy"
sad_songs_df["label"] = "sad"

# Merge happy and sad datasets
final_df = pd.concat([happy_songs_df, sad_songs_df], ignore_index=True)
final_df.to_csv("songs_dataset.csv", index=False)